In [ ]:
from langchain_ollama import ChatOllama

# Best general model: strong chat, instruction following, and tool calling.
MODEL_QWEN3_8B = "qwen3:8b"
# Good general-purpose alternative for chat, planning, and explanations.
MODEL_LLAMA31_8B = "llama3.1:8b"
# Good conversational and writing model with balanced speed and quality.
MODEL_MISTRAL_7B = "mistral:7b"
# Fast lightweight general model for simple questions and low latency.
MODEL_QWEN25_3B = "qwen2.5:3b"
# Fast lightweight chat model for simple agents and everyday tasks.
MODEL_LLAMA32_3B = "llama3.2:3b"
# Best coding model in this collection: code generation, debugging, and APIs.
MODEL_QWEN25_CODER_7B = "qwen2.5-coder:7b"
# Best reasoning-focused model: math, logic, and multi-step analysis; slower.
MODEL_DEEPSEEK_R1_7B = "deepseek-r1:7b"
# Compact general model with good instruction following.
MODEL_GEMMA3_4B = "gemma3:4b"
# Small reasoning/instruction model; useful when conserving VRAM.
MODEL_PHI4_MINI = "phi4-mini:latest"
# Fastest chat/agent model here; lower quality on complex tasks.
MODEL_LLAMA32_1B = "llama3.2:1b"
# Small Qwen model for fast lightweight prompts and classification.
MODEL_QWEN25_15B = "qwen2.5:1.5b"
# Vision model for image and screenshot understanding; not text-only chat.
MODEL_LLAVA_7B = "llava:7b"
# Embedding model for RAG/vector search; do not use as a chat model.
MODEL_NOMIC_EMBED_TEXT = "nomic-embed-text:latest"

# Change this one value to switch the chat model used below.
MODEL_NAME = MODEL_PHI4_MINI
OLLAMA_URL = "http://127.0.0.1:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=0,
)

print(f"Connected to Ollama. Using local model: {MODEL_NAME}")

In [ ]:
import os
from pathlib import Path
from typing import Annotated
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
import fitz
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langsmith import wrappers
from langsmith import Client, traceable
for env_path in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "Langgraph_tools" / "chatbot" / ".env",
]:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"
os.environ["LANGSMITH_PROJECT"] = "rag-evaluation"

In [ ]:
# Create the data points
from langsmith import Client

client=Client()

# Define the dataset - test data

dataset_name="Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)


In [ ]:
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."
@traceable(name="correctness_evaluator")
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    messages = [
        ("system", eval_instructions),
        ("human", user_content),
    ]
    response = llm.invoke(messages)
    return "CORRECT" in response.content

In [ ]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))



In [ ]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
@traceable(name="my_app")
def my_app(question: str, model: str = MODEL_NAME, instructions: str = default_instructions) -> str:
    messages = [
        ("system", instructions),
        ("human", question),
    ]
    response = llm.invoke(messages)
    return response.content

In [ ]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix=MODEL_NAME+f"-chatbot"
)

In [ ]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model=MODEL_PHI4_MINI)}
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix=MODEL_PHI4_MINI+f"-chatbot"
)